# 🔱 Shiv AI Voice Cloning v4.0
**Shri Ram Nag | PAISAWALA** — T4 GPU select karein pehle!

**Fixes:** Hakla issue ✅ | Voice Design ✅ | Instruct Mode ✅ | Modern UI ✅

In [ ]:
# STEP 1 — GPU + Install
import torch
assert torch.cuda.is_available(), '❌ GPU nahi mila! Runtime → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
!pip install -q gradio huggingface_hub transformers accelerate scipy
print('✅ Ready!')

In [ ]:
# STEP 2 — Download Model (~3.27 GB)
import os
from huggingface_hub import snapshot_download
LOCAL = './Shiv-AI-Voice-Cloning'
if not os.path.exists(LOCAL) or not os.listdir(LOCAL):
    print('📥 Downloading Shriramnag/Shiv-AI-Voice-Cloning ...')
    snapshot_download('Shriramnag/Shiv-AI-Voice-Cloning', local_dir=LOCAL, local_dir_use_symlinks=False)
else:
    print('✅ Already downloaded!')
print('Files:', [f for f in os.listdir(LOCAL)])

In [ ]:
# STEP 3 — Load Model
import sys, re, numpy as np, torch, gradio as gr
sys.path.insert(0, './Shiv-AI-Voice-Cloning')
from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name
try:
    from subtitle import LANGUAGE_CODE as WHISPER_LANGUAGE_CODE
except: pass

print('🔱 Loading model...')
model = OmniVoice.from_pretrained('./Shiv-AI-Voice-Cloning', device_map='cuda', dtype=torch.float16, load_asr=False)
SR = model.sampling_rate
print(f'✅ Loaded! SR={SR}')

In [ ]:
# STEP 4 — Core Logic (Chunking + All Functions)
import os; os.makedirs('./Shiv_Audio', exist_ok=True)

LANG_CHOICES = ['Auto'] + sorted(lang_display_name(n) for n in LANG_NAMES)
EVENT_TAGS   = ['[laughter]','[sigh]','[confirmation-en]','[question-en]','[surprise-wa]','[dissatisfaction-hnn]']
INSTRUCT_EX  = [
    'Speak slowly and clearly with a calm, deep voice',
    'Speak with excitement and high energy',
    'Speak softly like a bedtime story narrator',
    'Speak like a professional news anchor, formal and clear',
    'Speak in a sad, emotional tone with pauses',
    'Fast and enthusiastic like a radio jockey',
    'धीरे, शांत और गहरी आवाज़ में बोलें',
    'जोश और उत्साह के साथ तेज़ आवाज़ में बोलें',
]
INSERT_TAG_JS = """
(tag_val, current_text) => {
    const ta = document.querySelector('.shiv-tb textarea');
    if (!ta) return current_text + ' ' + tag_val;
    const s = ta.selectionStart, e = ta.selectionEnd;
    return current_text.slice(0,s) + ' ' + tag_val + ' ' + current_text.slice(e);
}
"""

def split_chunks(text, max_ch=120):
    lines = re.split(r'(?:\u2026\n?|\u0964\n|\n)', text)
    lines = [l.strip() for l in lines if l.strip()]
    chunks, cur = [], ''
    for line in lines:
        if len(line) > max_ch:
            if cur: chunks.append(cur); cur = ''
            for sent in re.split(r'(?<=[\u0964.!?])\s+', line):
                if len(cur)+len(sent)+1 <= max_ch: cur = (cur+' '+sent).strip()
                else:
                    if cur: chunks.append(cur)
                    cur = sent.strip()
        else:
            if len(cur)+len(line)+1 <= max_ch: cur = (cur+' '+line).strip()
            else:
                if cur: chunks.append(cur)
                cur = line.strip()
    if cur: chunks.append(cur)
    return [c for c in chunks if c.strip()]

def join_chunks(audios, silence_ms=0):
    if silence_ms > 0:
        sil = np.zeros(int(SR*silence_ms/1000), dtype=np.float32)
        parts = []
        for i,a in enumerate(audios):
            parts.append(a)
            if i < len(audios)-1: parts.append(sil)
        return np.concatenate(parts)
    return np.concatenate(audios)

def make_cfg(steps=32, gs=2.0, speed=1.0, pitch=0, energy=1.0):
    try: return OmniVoiceGenerationConfig(num_step=steps, guidance_scale=gs, denoise=True, preprocess_prompt=True, postprocess_output=True, speed=speed, pitch=pitch, energy=energy)
    except TypeError: return OmniVoiceGenerationConfig(num_step=steps, guidance_scale=gs, denoise=True, preprocess_prompt=True, postprocess_output=True)

def run_chunk(text, lang, cfg, vcp=None, instruct=None):
    kw = dict(text=text, language=lang if lang!='Auto' else None, generation_config=cfg)
    if vcp: kw['voice_clone_prompt'] = vcp
    if instruct: kw['instruct'] = instruct
    return model.generate(**kw)[0]

def to_wav(a): return (SR, (a*32767).astype(np.int16))

def fn_clone(text, lang, ref, ref_text):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    if not ref: return None, '⚠️ Reference audio upload karein'
    try:
        vcp = model.create_voice_clone_prompt(ref_audio=ref, ref_text=ref_text.strip() or None)
        cfg = make_cfg(); chunks = split_chunks(text)
        audio = join_chunks([run_chunk(c,lang,cfg,vcp=vcp) for c in chunks], silence_ms=0)
        return to_wav(audio), f'✅ {len(chunks)} chunks | {len(audio)/SR:.1f}s'
    except Exception as e: return None, f'❌ {e}'

def fn_design(text, lang, speed, pitch, energy, pause_ms, style):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    try:
        cfg = make_cfg(gs=2.5, speed=speed, pitch=pitch, energy=energy)
        chunks = split_chunks(text)
        audio = join_chunks([run_chunk(c,lang,cfg,instruct=style.strip() or None) for c in chunks], silence_ms=int(pause_ms))
        return to_wav(audio), f'✅ speed={speed} pitch={pitch} energy={energy} | {len(audio)/SR:.1f}s'
    except Exception as e:
        try:
            cfg = make_cfg(gs=2.5); chunks = split_chunks(text)
            audio = join_chunks([run_chunk(c,lang,cfg,instruct=style.strip() or None) for c in chunks])
            return to_wav(audio), f'✅ basic | {len(audio)/SR:.1f}s'
        except Exception as e2: return None, f'❌ {e2}'

def fn_tts(text, lang, steps, gs):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    try:
        cfg = make_cfg(int(steps), float(gs)); chunks = split_chunks(text)
        audio = join_chunks([run_chunk(c,lang,cfg) for c in chunks])
        return to_wav(audio), f'✅ {len(chunks)} chunks | {len(audio)/SR:.1f}s'
    except Exception as e: return None, f'❌ {e}'

def fn_instruct(text, lang, prompt):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    if not prompt or not prompt.strip(): return None, '⚠️ Instruction likhein'
    try:
        cfg = make_cfg(gs=3.0); chunks = split_chunks(text)
        audio = join_chunks([run_chunk(c,lang,cfg,instruct=prompt.strip()) for c in chunks])
        return to_wav(audio), f'✅ {len(chunks)} chunks | {len(audio)/SR:.1f}s'
    except Exception as e: return None, f'❌ {e}'

print('✅ Functions ready!')

In [ ]:
# STEP 5 — Launch Modern UI
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@600;800&family=DM+Sans:wght@300;400;500&display=swap');
:root{--saffron:#FF6B00;--gold:#FFB347;--dark:#0D0D0F;--surface:#16161A;--card:rgba(255,255,255,0.04);--border:rgba(255,107,0,0.2);--text:#F0EEE8;--muted:#888880;--radius:14px;}
*{box-sizing:border-box;} body,.gradio-container{background:var(--dark)!important;color:var(--text)!important;font-family:'DM Sans',sans-serif!important;} footer{display:none!important;} .gradio-container{max-width:100%!important;padding:0!important;}
.shiv-header{background:linear-gradient(135deg,#0D0D0F 0%,#1a0d00 50%,#0D0D0F 100%);border-bottom:1px solid var(--border);text-align:center;padding:28px 20px 20px;position:relative;overflow:hidden;}
.shiv-header::before{content:'';position:absolute;inset:0;background:radial-gradient(ellipse 60% 80% at 50% -20%,rgba(255,107,0,0.15),transparent);pointer-events:none;}
.shiv-header h1{font-family:'Syne',sans-serif!important;font-size:clamp(1.8em,4vw,2.8em);font-weight:800;background:linear-gradient(90deg,var(--saffron),var(--gold),var(--saffron));background-size:200%;-webkit-background-clip:text;-webkit-text-fill-color:transparent;animation:shimmer 3s linear infinite;margin:0 0 6px;}
@keyframes shimmer{to{background-position:200% center;}}
.shiv-header p{color:var(--muted);margin:3px 0;font-size:0.85em;} .shiv-header b{color:var(--gold);}
.tab-nav{background:var(--surface)!important;border-bottom:1px solid var(--border)!important;padding:0 16px!important;}
.tab-nav button{font-family:'DM Sans',sans-serif!important;font-weight:500!important;color:var(--muted)!important;border:none!important;border-bottom:2px solid transparent!important;padding:12px 16px!important;border-radius:0!important;background:transparent!important;transition:all .2s!important;}
.tab-nav button.selected,.tab-nav button:hover{color:var(--saffron)!important;border-bottom-color:var(--saffron)!important;}
.tabitem{padding:20px!important;}
label,.label-wrap span{font-family:'DM Sans',sans-serif!important;font-size:0.75em!important;font-weight:500!important;letter-spacing:0.08em!important;text-transform:uppercase!important;color:var(--gold)!important;}
.shiv-tb textarea,textarea{background:rgba(255,255,255,0.04)!important;border:1px solid var(--border)!important;border-radius:10px!important;color:var(--text)!important;font-family:'DM Sans',sans-serif!important;font-size:0.95em!important;padding:12px!important;transition:border-color .2s!important;}
.shiv-tb textarea:focus,textarea:focus{border-color:var(--saffron)!important;box-shadow:0 0 0 3px rgba(255,107,0,0.12)!important;outline:none!important;}
.wrap .wrap-inner,select{background:rgba(255,255,255,0.04)!important;border:1px solid var(--border)!important;border-radius:10px!important;color:var(--text)!important;}
input[type=range]{accent-color:var(--saffron)!important;}
.btn-primary{background:linear-gradient(135deg,var(--saffron),#e65000)!important;color:#fff!important;border:none!important;border-radius:10px!important;font-family:'Syne',sans-serif!important;font-weight:600!important;font-size:1em!important;padding:13px 24px!important;cursor:pointer!important;transition:transform .15s,box-shadow .15s!important;box-shadow:0 4px 20px rgba(255,107,0,0.35)!important;width:100%!important;}
.btn-primary:hover{transform:translateY(-2px)!important;box-shadow:0 8px 28px rgba(255,107,0,0.5)!important;}
.tag-btn{background:rgba(255,107,0,0.1)!important;border:1px solid rgba(255,107,0,0.3)!important;color:var(--gold)!important;border-radius:6px!important;font-size:0.72em!important;padding:4px 8px!important;transition:all .15s!important;}
.tag-btn:hover{background:rgba(255,107,0,0.25)!important;border-color:var(--saffron)!important;}
.preset-btn{background:rgba(255,255,255,0.05)!important;border:1px solid rgba(255,255,255,0.1)!important;color:var(--text)!important;border-radius:8px!important;font-size:0.82em!important;padding:7px 12px!important;transition:all .15s!important;}
.preset-btn:hover{background:rgba(255,107,0,0.15)!important;border-color:var(--saffron)!important;color:var(--gold)!important;}
.ex-btn{background:transparent!important;border:1px solid rgba(255,255,255,0.08)!important;color:var(--muted)!important;border-radius:6px!important;font-size:0.78em!important;padding:5px 10px!important;text-align:left!important;width:100%!important;transition:all .15s!important;margin-bottom:4px!important;}
.ex-btn:hover{border-color:var(--saffron)!important;color:var(--gold)!important;background:rgba(255,107,0,0.08)!important;}
.status-box textarea{background:rgba(0,0,0,0.3)!important;border:1px solid rgba(255,255,255,0.06)!important;color:var(--gold)!important;font-size:0.82em!important;border-radius:8px!important;}
.audio-wrap{background:rgba(255,107,0,0.06)!important;border:1px solid var(--border)!important;border-radius:var(--radius)!important;}
.info-card{background:rgba(255,107,0,0.06);border:1px solid rgba(255,107,0,0.15);border-radius:10px;padding:14px 16px;margin-top:12px;font-size:0.85em;color:var(--muted);line-height:1.7;}
.info-card strong{color:var(--gold);}
.sec-title{font-family:'Syne',sans-serif;font-size:1.05em;font-weight:700;color:var(--gold);margin:0 0 12px;}
.divider{border:none;border-top:1px solid var(--border);margin:14px 0;}
.shiv-footer{text-align:center;padding:18px;color:var(--muted);font-size:0.8em;border-top:1px solid var(--border);background:var(--surface);}
.shiv-footer span{color:var(--saffron);}
"""

with gr.Blocks(css=CSS, title='🔱 Shiv AI Voice Cloning') as demo:
    gr.HTML("""
    <div class='shiv-header'>
      <h1>🔱 Shiv AI Voice Cloning</h1>
      <p>Advanced Multilingual Neural Speech Engine &nbsp;·&nbsp; 646 Languages</p>
      <p><b>Shri Ram Nag</b> &nbsp;·&nbsp; PAISAWALA 🎬 &nbsp;·&nbsp; v4.0</p>
    </div>""")

    with gr.Tabs(elem_classes='tab-nav'):

        # TAB 1 — Voice Clone
        with gr.TabItem('🎙️ Voice Clone'):
            with gr.Row():
                with gr.Column(scale=6):
                    gr.HTML('<div class="sec-title">📝 Script</div>')
                    vc_text = gr.Textbox(lines=8, elem_classes='shiv-tb', placeholder='पूरी script paste करें…', label='', show_label=False)
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b = gr.Button(tag, elem_classes='tag-btn', size='sm'); b.click(fn=None, inputs=[b,vc_text], outputs=vc_text, js=INSERT_TAG_JS)
                    with gr.Row():
                        vc_lang = gr.Dropdown(LANG_CHOICES, value='Auto', label='🌐 Language', scale=1)
                        vc_ref_text = gr.Textbox(label='📄 Reference Transcript (optional)', lines=1, scale=2)
                    vc_ref = gr.Audio(label='🎤 Reference Audio', type='filepath')
                    vc_btn = gr.Button('🔱  Clone Voice & Generate', elem_classes='btn-primary')
                with gr.Column(scale=5):
                    gr.HTML('<div class="sec-title">🔊 Output</div>')
                    vc_out = gr.Audio(type='numpy', label='', elem_classes='audio-wrap')
                    vc_status = gr.Textbox(label='Status', interactive=False, elem_classes='status-box')
                    gr.HTML('<div class="info-card"><strong>✅ Hakla issue fixed</strong><br>Chunks join seamlessly — no extra silence<br><br><strong>💡 Tips:</strong> 5–30 sec saaf audio | <code>…</code> = natural pause</div>')
            vc_btn.click(fn_clone, [vc_text,vc_lang,vc_ref,vc_ref_text], [vc_out,vc_status])

        # TAB 2 — Voice Design
        with gr.TabItem('🎛️ Voice Design'):
            with gr.Row():
                with gr.Column(scale=6):
                    gr.HTML('<div class="sec-title">📝 Script</div>')
                    vd_text = gr.Textbox(lines=6, elem_classes='shiv-tb', placeholder='यहाँ text…', label='', show_label=False)
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b2 = gr.Button(tag, elem_classes='tag-btn', size='sm'); b2.click(fn=None, inputs=[b2,vd_text], outputs=vd_text, js=INSERT_TAG_JS)
                    vd_lang   = gr.Dropdown(LANG_CHOICES, value='Auto', label='🌐 Language')
                    gr.HTML('<hr class="divider"><div class="sec-title">🎚️ Controls</div>')
                    vd_speed  = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label='⚡ Speed')
                    vd_pitch  = gr.Slider(-12, 12,  value=0,   step=1,    label='🎵 Pitch')
                    vd_energy = gr.Slider(0.3, 2.0, value=1.0, step=0.05, label='💪 Energy')
                    vd_pause  = gr.Slider(0,   400, value=0,   step=50,   label='⏸️ Extra Pause (ms)')
                    vd_style  = gr.Textbox(label='✍️ Style Instruction', lines=2)
                    gr.HTML('<hr class="divider"><div class="sec-title">⚡ Presets</div>')
                    with gr.Row():
                        pc=gr.Button('😌 Calm',elem_classes='preset-btn'); pe=gr.Button('🔥 Excited',elem_classes='preset-btn')
                        pn=gr.Button('📺 News',elem_classes='preset-btn'); ps=gr.Button('📖 Story',elem_classes='preset-btn')
                    vd_btn = gr.Button('🎛️  Design & Generate', elem_classes='btn-primary')
                with gr.Column(scale=5):
                    gr.HTML('<div class="sec-title">🔊 Output</div>')
                    vd_out = gr.Audio(type='numpy', label='', elem_classes='audio-wrap')
                    vd_status = gr.Textbox(label='Status', interactive=False, elem_classes='status-box')
                    gr.HTML('<div class="info-card"><strong>Controls:</strong><br>Speed ↓=slow ↑=fast | Pitch ↓=deep ↑=high<br>Energy ↓=soft ↑=loud | Pause=silence btwn chunks</div>')
            pc.click(fn=lambda:(0.8,-2,0.7,0,'speak calmly'),   outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            pe.click(fn=lambda:(1.3,3,1.5,0,'speak with excitement'), outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            pn.click(fn=lambda:(1.0,0,1.1,0,'speak like a news anchor, formal'), outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            ps.click(fn=lambda:(0.85,-1,0.8,100,'speak like a warm storyteller'), outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            vd_btn.click(fn_design,[vd_text,vd_lang,vd_speed,vd_pitch,vd_energy,vd_pause,vd_style],[vd_out,vd_status])

        # TAB 3 — Simple TTS
        with gr.TabItem('🔤 Simple TTS'):
            with gr.Row():
                with gr.Column(scale=6):
                    gr.HTML('<div class="sec-title">📝 Script</div>')
                    tts_text = gr.Textbox(lines=8, elem_classes='shiv-tb', placeholder='Text paste karein…', label='', show_label=False)
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b3=gr.Button(tag,elem_classes='tag-btn',size='sm'); b3.click(fn=None,inputs=[b3,tts_text],outputs=tts_text,js=INSERT_TAG_JS)
                    tts_lang = gr.Dropdown(LANG_CHOICES, value='Auto', label='🌐 Language')
                    with gr.Row():
                        tts_steps=gr.Slider(10,64,value=40,step=2,label='🔢 Steps (quality)',scale=1)
                        tts_gs   =gr.Slider(1.0,5.0,value=3.0,step=0.5,label='🎯 Guidance (realism)',scale=1)
                    tts_btn = gr.Button('🔤  Generate HD Audio', elem_classes='btn-primary')
                with gr.Column(scale=5):
                    gr.HTML('<div class="sec-title">🔊 Output</div>')
                    tts_out=gr.Audio(type='numpy',label='',elem_classes='audio-wrap')
                    tts_status=gr.Textbox(label='Status',interactive=False,elem_classes='status-box')
                    gr.HTML('<div class="info-card"><strong>HD Tips:</strong><br>Steps=40, Guidance=3.0 → best realism<br>Steps=60, Guidance=4.0 → max quality</div>')
            tts_btn.click(fn_tts,[tts_text,tts_lang,tts_steps,tts_gs],[tts_out,tts_status])

        # TAB 4 — Instruct Mode
        with gr.TabItem('📋 Instruct Mode'):
            with gr.Row():
                with gr.Column(scale=6):
                    gr.HTML('<div class="sec-title">📝 Script</div>')
                    inst_text=gr.Textbox(lines=6,elem_classes='shiv-tb',placeholder='यहाँ text…',label='',show_label=False)
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b4=gr.Button(tag,elem_classes='tag-btn',size='sm'); b4.click(fn=None,inputs=[b4,inst_text],outputs=inst_text,js=INSERT_TAG_JS)
                    inst_lang   = gr.Dropdown(LANG_CHOICES,value='Auto',label='🌐 Language')
                    inst_prompt = gr.Textbox(label='📋 Style Instruction',placeholder='Speak slowly and clearly…',lines=3)
                    gr.HTML('<hr class="divider"><div class="sec-title">💡 Examples</div>')
                    for ex in INSTRUCT_EX:
                        eb=gr.Button(ex,elem_classes='ex-btn',size='sm'); eb.click(fn=lambda x=ex:x,outputs=inst_prompt)
                    inst_btn=gr.Button('📋  Generate with Instruction',elem_classes='btn-primary')
                with gr.Column(scale=5):
                    gr.HTML('<div class="sec-title">🔊 Output</div>')
                    inst_out=gr.Audio(type='numpy',label='',elem_classes='audio-wrap')
                    inst_status=gr.Textbox(label='Status',interactive=False,elem_classes='status-box')
                    gr.HTML('<div class="info-card"><strong>Tips:</strong><br>English instructions best work karte hain<br><em>"Speak like a Bollywood trailer narrator"</em><br><em>"Soft and emotional like reading a love letter"</em></div>')
            inst_btn.click(fn_instruct,[inst_text,inst_lang,inst_prompt],[inst_out,inst_status])

    gr.HTML("<div class='shiv-footer'>© 2026 <span>🔱 Shiv AI</span> &nbsp;·&nbsp; <span>Shri Ram Nag</span> &nbsp;·&nbsp; PAISAWALA 🎬</div>")

demo.launch(share=True, debug=False)
print('🔱 Shiv AI v4.0 launched!')